# syndrome-mitigation-qec — larger-shot / d=7 run (Colab, CPU)

Independent toy re-implementation of **Virtual Error Cancellation**
([arXiv:2610.12400](https://arxiv.org/abs/2610.12400), Shtanko, Roy, Kumar, Minev,
Google Quantum AI, CC BY 4.0). This notebook re-runs the pre-registered pipeline with
**larger shot counts and a d=7 primary-style run** than the in-repo CPU experiment.

**Runtime note:** this runs fine on the **free CPU Colab tier** (the full cell budget below
is ~10–20 min). It needs no GPU. **Do not switch to a paid or GPU runtime without the
repo owner's approval** — there is nothing here a GPU would accelerate anyway
(stim + PyMatching + numpy are CPU codes).

*This notebook is committed unexecuted.*

In [ ]:
!pip -q install numpy stim pymatching matplotlib pyyaml scipy
!git clone -q https://github.com/Ryukijano/tech-demos.git
%cd tech-demos/apps/syndrome-mitigation-qec

## 1. Verify the exact ML decoder against brute force (d=3) and time d=5/d=7

In [ ]:
!python src/verify.py

## 2. Larger-shot d=7 run (seed 0/1/2)

The in-repo experiment treats d=7 as report-only with 2e5 shots. Here we give it
primary-style treatment: 3 seeds, 1e6 calibration + 1e6 eval shots each.
Expect (from the paper's Fig. 2, QUOTED, not a promise): w_I*≈1.01,
w_X*=w_Z*≈−0.42, w_Y*≈−4e-2 at p=0.05, and η well below 2.

In [ ]:
import sys, json, numpy as np
sys.path.insert(0, ".")
from src.run_experiment import run_one

for seed in (0, 1, 2):
    out, _ = run_one(d=7, p=0.05, seed=seed, n_cal=1_000_000, n_eval=1_000_000,
                     cal_shards=1000, eval_shards=20,
                     cal_seed=10700 + seed, eval_seed=20700 + seed)
    print(f"d=7 seed {seed}: p_L={out['p_l_mwpm']:.3e}  "
          f"p_res_PS={out['p_res_postselect']:.3e}  "
          f"p_res_VEC={out['p_res_vec']:.3e}±{out['p_res_vec_se']:.1e}  "
          f"C={out['C']:.4f}  eta={out['eta']:.2f}")
    print("   weights (I,X,Y,Z):", [round(w, 4) for w in out['weights_IXYZ']])

## 3. Optional: denser p-sweep at d=7

Each point costs roughly a minute at low p (more at high p where more distinct
syndromes must be contracted). Reduce `p_values` if the session is slow.

In [ ]:
p_values = [0.02, 0.03, 0.05, 0.08, 0.11]
rows = []
for p in p_values:
    out, _ = run_one(d=7, p=p, seed=0, n_cal=300_000, n_eval=300_000,
                     cal_shards=20, eval_shards=20,
                     cal_seed=10750, eval_seed=20750)
    rows.append((p, out['p_l_mwpm'], out.get('p_res_vec'), out.get('eta')))
    print(rows[-1])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
rows_ok = [r for r in rows if r[2] is not None]
p, pl, pres, eta = map(np.array, zip(*rows_ok))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].loglog(p, pl, 'o-', label='$p_L$ (MWPM)')
ax[0].loglog(p, pres, 's-', label='$p_{res}$ VEC')
ax[0].set_xlabel('p'); ax[0].legend(); ax[0].set_title('d=7 residual bias')
ax[1].loglog(p, eta, 'o-'); ax[1].axhline(4, ls='--', c='k')
ax[1].axhline(2, ls=':', c='gray')
ax[1].set_xlabel('p'); ax[1].set_title('$\\eta=(C-1)/p_L$ vs PEC ($\\eta$=4)')
plt.tight_layout(); plt.show()